# DBiT-seq dataset 30258697

**Publication:** [Spatial CUT&Tag — Nature Methods 2024](https://www.nature.com/articles/s41592-024-02576-0)

Mouse E13 embryo, multi-modal spatial (RNA + histone marks + ATAC).

Three sample groups:
- **E13_20_fig3** (20 µm): RNA, H3K27me3, H3K4me3
- **E13_50_2** (50 µm): RNA, ATAC, H3K27me3, H3K4me3
- **E13_50_1** (50 µm): H3K27ac, H3K27me3 (no RNA)

Goal: combine modalities per sample into single h5ad files with modalities as layers.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
import matplotlib.pyplot as plt

DATA_DIR = Path('/Volumes/processing2/30258697')
OUT_DIR = Path('/Volumes/processing2/KaroSpaceDataWrangle/dbit-30258697')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f'Data dir not found: {DATA_DIR}'

## 1. File discovery & grouping

In [ ]:
h5ad_files = sorted(DATA_DIR.glob('*.h5ad'))
print(f'Found {len(h5ad_files)} h5ad files:\n')

# Group by sample prefix
from collections import defaultdict
groups = defaultdict(dict)

for f in h5ad_files:
    name = f.stem
    # Split into sample + modality
    if name.startswith('E13_20_fig3_'):
        sample = 'E13_20_fig3'
        modality = name.replace('E13_20_fig3_', '')
    elif name.startswith('E13_50_2_'):
        sample = 'E13_50_2'
        modality = name.replace('E13_50_2_', '')
    elif name.startswith('E13_50_1_'):
        sample = 'E13_50_1'
        modality = name.replace('E13_50_1_', '')
    else:
        print(f'  Skipping: {name}')
        continue
    groups[sample][modality] = f
    print(f'  {sample} / {modality}: {f.name}')

print()
for sample, mods in groups.items():
    print(f'{sample}: {list(mods.keys())}')

## 2. Explore each sample group

In [ ]:
for sample, mods in groups.items():
    print(f'\n{"=" * 60}')
    print(f'Sample: {sample}')
    print(f'{"=" * 60}')
    for modality, path in mods.items():
        a = sc.read_h5ad(path)
        sp_keys = list(a.obsm.keys())
        is_sparse = sparse.issparse(a.X)
        xmax = a.X.max() if is_sparse else np.max(a.X)
        xmin = a.X.min() if is_sparse else np.min(a.X)
        print(f'\n  {modality}:')
        print(f'    Shape: {a.shape}')
        print(f'    Sparse: {is_sparse}')
        print(f'    X range: [{xmin:.2f}, {xmax:.2f}]')
        print(f'    obs cols: {list(a.obs.columns)}')
        print(f'    obsm keys: {sp_keys}')
        print(f'    var names unique: {a.var_names.is_unique}')
        print(f'    var[:5]: {list(a.var_names[:5])}')
        # Show spatial coord range
        for sk in sp_keys:
            coords = a.obsm[sk]
            print(f'    {sk} range: x=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], '
                  f'y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}]')
        del a

## 3. Check barcode and gene overlap within groups

In [ ]:
for sample, mods in groups.items():
    print(f'\n--- {sample} ---')
    obs_sets = {}
    var_sets = {}
    for modality, path in mods.items():
        a = sc.read_h5ad(path)
        a.var_names_make_unique()
        obs_sets[modality] = set(a.obs_names)
        var_sets[modality] = set(a.var_names)
        del a

    # Barcode overlap
    mod_names = list(obs_sets.keys())
    common_obs = obs_sets[mod_names[0]]
    for m in mod_names[1:]:
        common_obs = common_obs & obs_sets[m]
    print(f'  Barcodes per modality: {{", ".join(f"{m}: {len(obs_sets[m])}" for m in mod_names)}}')
    print(f'  Common barcodes: {len(common_obs)}')

    # Gene overlap (only meaningful between epigenetic modalities or RNA vs epigenetic)
    for i, m1 in enumerate(mod_names):
        for m2 in mod_names[i+1:]:
            overlap = var_sets[m1] & var_sets[m2]
            print(f'  Gene overlap {m1}({len(var_sets[m1])}) ∩ {m2}({len(var_sets[m2])}): {len(overlap)}')

## 4. Visualize spatial layout per sample

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (sample, mods) in zip(axes, groups.items()):
    # Use first modality to show spatial layout
    first_mod = list(mods.keys())[0]
    a = sc.read_h5ad(mods[first_mod])
    # Pick spatial key
    if 'spatial' in a.obsm:
        coords = a.obsm['spatial']
    elif 'spatial_pxl' in a.obsm:
        coords = a.obsm['spatial_pxl']
    else:
        coords = list(a.obsm.values())[0]

    ax.scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.5)
    ax.set_title(f'{sample}\n({a.n_obs} spots, {first_mod})')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    del a

plt.tight_layout()
plt.show()

## 5. Combine modalities per sample

Strategy:
- Find common barcodes across all modalities in a sample
- Find common genes between epigenetic modalities (they share 24,333 genes)
- For samples with RNA: use all RNA genes as `X` + `layers['RNA']`, subset epigenetic to common genes and store as layers
- For samples without RNA (E13_50_1): use first modality as `X`, others as layers
- Unify spatial coords → `obsm['spatial']`

In [ ]:
def combine_sample(sample_id, mod_paths):
    """Combine modalities for one sample into a single AnnData.

    RNA (if present) becomes X with all its genes.
    Epigenetic modalities stored as layers, subset to their own gene set.
    """
    # Load all modalities
    adatas = {}
    for mod, path in mod_paths.items():
        a = sc.read_h5ad(path)
        a.var_names_make_unique()
        adatas[mod] = a

    # Find common barcodes
    common_bc = None
    for a in adatas.values():
        bc = set(a.obs_names)
        common_bc = bc if common_bc is None else common_bc & bc
    common_bc = sorted(common_bc)
    print(f'  Common barcodes: {len(common_bc)}')

    # Determine primary modality (RNA if available, else first)
    has_rna = 'RNA' in adatas
    if has_rna:
        primary_mod = 'RNA'
    else:
        primary_mod = list(adatas.keys())[0]

    primary = adatas[primary_mod][common_bc].copy()

    # Get spatial coords
    if 'spatial' in primary.obsm:
        spatial = primary.obsm['spatial']
    elif 'spatial_pxl' in primary.obsm:
        spatial = primary.obsm['spatial_pxl']
    else:
        spatial = list(primary.obsm.values())[0]

    # Store primary as a layer too
    if sparse.issparse(primary.X):
        primary.layers[primary_mod] = primary.X.copy()
    else:
        primary.layers[primary_mod] = primary.X.copy()

    # Add other modalities as layers
    # Epigenetic modalities have different var (24,333 genes vs 48,440 for RNA)
    # Store epigenetic layers aligned to their own genes → use obsm for now,
    # and also create a version with only shared genes as layers
    epi_mods = {m: a for m, a in adatas.items() if m != primary_mod}
    epi_gene_info = {}  # Track gene names per epigenetic modality

    if has_rna and epi_mods:
        # Find genes shared between RNA and epigenetic modalities
        rna_genes = set(primary.var_names)
        for mod, a in epi_mods.items():
            epi_a = a[common_bc].copy()
            epi_genes = set(epi_a.var_names)
            shared = sorted(rna_genes & epi_genes)
            print(f'  {mod}: {len(epi_genes)} genes, {len(shared)} shared with RNA')
            epi_gene_info[mod] = list(epi_a.var_names)

            # Store full epigenetic matrix in obsm (preserves all features)
            if sparse.issparse(epi_a.X):
                primary.obsm[f'X_{mod}'] = np.array(epi_a[:, :].X.toarray(), dtype=np.float32)
            else:
                primary.obsm[f'X_{mod}'] = np.array(epi_a.X, dtype=np.float32)

            # Store shared-gene subset as layer (aligned to RNA var)
            rna_idx = [list(primary.var_names).index(g) for g in shared]
            layer = np.zeros((primary.n_obs, primary.n_vars), dtype=np.float32)
            epi_shared = epi_a[:, shared]
            if sparse.issparse(epi_shared.X):
                layer[:, rna_idx] = epi_shared.X.toarray()
            else:
                layer[:, rna_idx] = epi_shared.X
            primary.layers[mod] = sparse.csr_matrix(layer)
            del epi_a
    else:
        # No RNA — all modalities share same var (24,333 genes)
        for mod, a in epi_mods.items():
            epi_a = a[common_bc].copy()
            # Reindex to match primary var
            epi_a = epi_a[:, primary.var_names]
            if sparse.issparse(epi_a.X):
                primary.layers[mod] = epi_a.X.copy()
            else:
                primary.layers[mod] = sparse.csr_matrix(epi_a.X)
            del epi_a

    # Clean up obsm — keep only spatial
    obsm_to_keep = {k: v for k, v in primary.obsm.items() if k.startswith('X_')}
    primary.obsm.clear()
    primary.obsm['spatial'] = np.array(spatial, dtype=np.float64)
    for k, v in obsm_to_keep.items():
        primary.obsm[k] = v

    # Store metadata
    primary.obs['sample_id'] = sample_id
    primary.uns['sample_id'] = sample_id
    primary.uns['modalities'] = list(mod_paths.keys())
    primary.uns['primary_modality'] = primary_mod
    if epi_gene_info:
        primary.uns['epi_gene_names'] = epi_gene_info

    # Clean obs columns
    keep_cols = ['sample_id']
    for c in ['in_tissue', 'array_row', 'array_col']:
        if c in primary.obs.columns:
            keep_cols.append(c)
    primary.obs = primary.obs[keep_cols]

    print(f'  Combined: {primary.shape}, layers: {list(primary.layers.keys())}')
    print(f'  obsm: {list(primary.obsm.keys())}')
    return primary

In [ ]:
combined = {}
for sample, mods in groups.items():
    print(f'\n--- Combining {sample} ---')
    combined[sample] = combine_sample(sample, mods)
    print()

## 6. Inspect combined objects

In [ ]:
for sample, adata in combined.items():
    print(f'\n{"=" * 50}')
    print(f'{sample}')
    print(f'{"=" * 50}')
    print(adata)
    print(f'\nLayers: {list(adata.layers.keys())}')
    print(f'obsm: {list(adata.obsm.keys())}')
    print(f'X range: [{adata.X.min():.2f}, {adata.X.max():.2f}]' if not sparse.issparse(adata.X) 
          else f'X range: [{adata.X.min():.2f}, {adata.X.max():.2f}]')
    for layer_name in adata.layers:
        L = adata.layers[layer_name]
        lmax = L.max() if sparse.issparse(L) else np.max(L)
        lmin = L.min() if sparse.issparse(L) else np.min(L)
        nnz = L.nnz if sparse.issparse(L) else np.count_nonzero(L)
        total = L.shape[0] * L.shape[1]
        print(f'  layer {layer_name}: range=[{lmin:.2f}, {lmax:.2f}], '
              f'nonzero={nnz}/{total} ({100*nnz/total:.1f}%)')

## 7. Spatial plots of combined data

In [ ]:
for sample, adata in combined.items():
    n_layers = len(adata.layers)
    fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 5))
    if n_layers == 1:
        axes = [axes]

    coords = adata.obsm['spatial']

    for ax, layer_name in zip(axes, adata.layers):
        L = adata.layers[layer_name]
        if sparse.issparse(L):
            total_signal = np.array(L.sum(axis=1)).flatten()
        else:
            total_signal = L.sum(axis=1)
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1], c=total_signal,
                            s=2, alpha=0.7, cmap='viridis')
        ax.set_title(f'{sample}\n{layer_name} total signal')
        ax.set_aspect('equal')
        ax.invert_yaxis()
        plt.colorbar(sc_plot, ax=ax, shrink=0.6)

    plt.tight_layout()
    plt.show()

## 8. Write combined h5ad files

In [ ]:
for sample, adata in combined.items():
    out_path = OUT_DIR / f'{sample}.h5ad'
    adata.write_h5ad(out_path)
    print(f'Wrote {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)')

## 9. Summary

In [ ]:
summary_rows = []
for sample, adata in combined.items():
    summary_rows.append({
        'sample': sample,
        'n_cells': adata.n_obs,
        'n_genes': adata.n_vars,
        'layers': ', '.join(adata.layers.keys()),
        'primary': adata.uns['primary_modality'],
        'modalities': ', '.join(adata.uns['modalities']),
    })

pd.DataFrame(summary_rows)